# Step-1: Loading the Datasets


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

train_df = pd.read_csv('Question Answer Classification Dataset 7.csv')
test_df = pd.read_csv('[Updated] Question Answer Classification Dataset[Test].csv')

print("Training set shape:", train_df.shape)
print("Testing set shape:", test_df.shape)

text_column = 'QA Text'
label_column = 'Class'


In [ ]:
print("\nTraining Data Sample")
print(train_df.head())

print("\nTest Data Sample")
print(test_df.head())

print("\nColumn Names")
print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

print("\nData Types")
print(train_df.dtypes)

print("\nDataset Info")
train_df.info()


In [ ]:
print("\nClass Distribution")
print(train_df['Class'].value_counts())

plt.figure(figsize=(10, 6))
train_df['Class'].value_counts().plot(kind='bar')
plt.title('Class Distribution in Training Set')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nClass Balance")
print(train_df['Class'].value_counts(normalize=True) * 100)


# Step-2: EDA

## Step-2.1: Basic Dataset statistics

In [ ]:
print("="*80)
print("EXPLORATORY DATA ANALYSIS")
print("="*80)

print("\n[1] DATASET OVERVIEW")
print("-"*80)
print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Features: {train_df.columns.tolist()}")
print(f"\nData types:\n{train_df.dtypes}")

print("\n[2] MISSING VALUES CHECK")
print("-"*80)
print("Training set missing values:")
print(train_df.isnull().sum())
print("\nTest set missing values:")
print(test_df.isnull().sum())

print("\n[3] DUPLICATE CHECK")
print("-"*80)
train_duplicates = train_df.duplicated().sum()
test_duplicates = test_df.duplicated().sum()
print(f"Training set duplicates: {train_duplicates}")
print(f"Test set duplicates: {test_duplicates}")

print("\n[4] SAMPLE DATA")
print("-"*80)
print(train_df.head())


## Step 2.2: Class distribution analysis

In [ ]:
print("\n[5] CLASS DISTRIBUTION ANALYSIS")
print("-"*80)

class_counts = train_df[label_column].value_counts()
class_percentages = train_df[label_column].value_counts(normalize=True) * 100

print("\nClass counts in training set:")
print(class_counts)
print("\nClass percentages:")
print(class_percentages)

max_class_count = class_counts.max()
min_class_count = class_counts.min()
imbalance_ratio = max_class_count / min_class_count

print(f"\nClass Balance Analysis:")
print(f"  Most frequent class: {class_counts.index[0]} ({max_class_count} samples)")
print(f"  Least frequent class: {class_counts.index[-1]} ({min_class_count} samples)")
print(f"  Imbalance ratio: {imbalance_ratio:.2f}:1")

if imbalance_ratio > 3:
    print("  Dataset is IMBALANCED - consider stratified sampling")
else:
    print("  Dataset is relatively balanced")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

class_counts.plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Class Distribution - Training Set', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

for i, v in enumerate(class_counts):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

axes[1].pie(class_counts, labels=class_counts.index, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 10})
axes[1].set_title('Class Distribution - Percentage', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n EDA FINDING 1: Class Distribution")
print(f"   - Number of classes: {train_df[label_column].nunique()}")
print(f"   - Balance status: {'Imbalanced' if imbalance_ratio > 3 else 'Balanced'}")


## Step 2.3: Text Length Analysis

In [ ]:
print("\n[6] TEXT LENGTH ANALYSIS")
print("-"*80)

train_df['char_count'] = train_df[text_column].astype(str).apply(len)
train_df['word_count'] = train_df[text_column].astype(str).apply(lambda x: len(x.split()))

print("Character count statistics:")
print(train_df['char_count'].describe())
print("\nWord count statistics:")
print(train_df['word_count'].describe())

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].hist(train_df['char_count'], bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribution of Character Count', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Number of Characters')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(train_df['char_count'].mean(), color='red', linestyle='--',
                    label=f'Mean: {train_df["char_count"].mean():.0f}')
axes[0, 0].legend()

axes[0, 1].hist(train_df['word_count'], bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution of Word Count', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Number of Words')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(train_df['word_count'].mean(), color='red', linestyle='--',
                    label=f'Mean: {train_df["word_count"].mean():.0f}')
axes[0, 1].legend()

train_df.boxplot(column='char_count', by=label_column, ax=axes[1, 0])
axes[1, 0].set_title('Character Count by Class', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Class')
axes[1, 0].set_ylabel('Character Count')
plt.sca(axes[1, 0])
plt.xticks(rotation=45, ha='right')

train_df.boxplot(column='word_count', by=label_column, ax=axes[1, 1])
axes[1, 1].set_title('Word Count by Class', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Class')
axes[1, 1].set_ylabel('Word Count')
plt.sca(axes[1, 1])
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

print("\nEDA FINDING 2: Text Length Characteristics")
print(f"   - Average character count: {train_df['char_count'].mean():.0f}")
print(f"   - Average word count: {train_df['word_count'].mean():.0f}")
print(f"   - Max character count: {train_df['char_count'].max()}")
print(f"   - Min character count: {train_df['char_count'].min()}")


## Step 2.4: HTML and Special Characters Analysis

In [ ]:
import re

print("\n[7] HTML & SPECIAL CHARACTERS ANALYSIS")
print("-"*80)

def contains_html(text):
    return bool(re.search(r'<[^>]+>', str(text)))

train_df['has_html'] = train_df[text_column].apply(contains_html)
html_count = train_df['has_html'].sum()
html_percentage = (html_count / len(train_df)) * 100

print(f"Texts containing HTML tags: {html_count} ({html_percentage:.2f}%)")

sample_text = train_df[text_column].iloc[0]
html_tags = re.findall(r'<[^>]+>', sample_text)
print(f"Sample HTML tags found: {set(html_tags)}")

def contains_url(text):
    return bool(re.search(r'http[s]?://|www\.', str(text)))

train_df['has_url'] = train_df[text_column].apply(contains_url)
url_count = train_df['has_url'].sum()
print(f"\nTexts containing URLs: {url_count} ({(url_count/len(train_df))*100:.2f}%)")

def special_char_ratio(text):
    special_chars = re.findall(r'[^a-zA-Z0-9\s]', str(text))
    return len(special_chars) / len(str(text)) if len(str(text)) > 0 else 0

train_df['special_char_ratio'] = train_df[text_column].apply(special_char_ratio)
print(f"\nAverage special character ratio: {train_df['special_char_ratio'].mean():.4f}")

def digit_ratio(text):
    digits = re.findall(r'\d', str(text))
    return len(digits) / len(str(text)) if len(str(text)) > 0 else 0

train_df['digit_ratio'] = train_df[text_column].apply(digit_ratio)
print(f"Average digit ratio: {train_df['digit_ratio'].mean():.4f}")

print("\n EDA FINDING 3: Text Characteristics")
print(f"   - HTML tags present: {html_percentage:.2f}% of texts")
print(f"   - URLs present: {(url_count/len(train_df))*100:.2f}% of texts")
print(f"   - Special characters: {train_df['special_char_ratio'].mean():.2%} average")
print(f"   - Digits: {train_df['digit_ratio'].mean():.2%} average")


In [ ]:
import nltk
nltk.download('all')


## Step 2.5: Vocabulary Analysis

In [ ]:
from collections import Counter
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize

print("\n[8] VOCABULARY ANALYSIS")
print("-"*80)

all_text = ' '.join(train_df[text_column].astype(str))
words = word_tokenize(all_text.lower())

unique_words = set(words)
print(f"Total words: {len(words)}")
print(f"Unique words (vocabulary size): {len(unique_words)}")

word_freq = Counter(words)
most_common = word_freq.most_common(20)
print(f"\nTop 20 most common words:")
for word, count in most_common:
    print(f"  '{word}': {count}")

top_words = dict(most_common[:15])
plt.figure(figsize=(12, 6))
plt.bar(top_words.keys(), top_words.values(), color='steelblue', edgecolor='black')
plt.title('Top 15 Most Frequent Words', fontsize=14, fontweight='bold')
plt.xlabel('Words', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nEDA FINDING 4: Vocabulary Characteristics")
print(f"   - Total vocabulary size: {len(unique_words)}")
print(f"   - Average word frequency: {len(words)/len(unique_words):.2f}")


## Step 2.6: Class-Specific Analysis

In [ ]:
print("\n[9] CLASS-SPECIFIC TEXT ANALYSIS")
print("-"*80)

class_stats = train_df.groupby(label_column).agg({
    'char_count': ['mean', 'std'],
    'word_count': ['mean', 'std']
}).round(2)

print("\nText statistics by class:")
print(class_stats)

print("\n" + "="*80)
print("SAMPLE TEXTS FROM EACH CLASS")
print("="*80)

for class_name in train_df[label_column].unique()[:3]:
    print(f"\n{'='*80}")
    print(f"CLASS: {class_name}")
    print(f"{'='*80}")
    sample = train_df[train_df[label_column] == class_name].iloc[0][text_column]
    print(sample[:400] + "...")

print("\nEDA FINDING 5: Class-wise Observations")
print("   - Different classes may have different average text lengths")
print("   - Some classes may use more technical vocabulary")


## Step 2.7: EDA Summary & Conclusions

In [ ]:
print("\n" + "="*80)
print("EDA SUMMARY & PREPROCESSING DECISIONS")
print("="*80)

print("\nKEY FINDINGS:")
print(f"  1. Dataset has {train_df[label_column].nunique()} classes")
print(f"  2. {'Imbalanced' if imbalance_ratio > 3 else 'Balanced'} class distribution")
print(f"  3. {html_percentage:.0f}% of texts contain HTML tags → NEED TO REMOVE")
print(f"  4. Average text length: {train_df['word_count'].mean():.0f} words")
print(f"  5. Vocabulary size: {len(unique_words)} unique words")
print(f"  6. Special characters present → NEED TO CLEAN")
print(f"  7. URLs present in some texts → NEED TO REMOVE")


## Step 2.8: Wordcloud

In [ ]:
!pip install wordcloud


# PHASE 3: PREPROCESSING IMPLEMENTATION

## Step 3.1: Create Preprocessing Functions

In [ ]:
import re
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

def clean_html(text):
    """Remove HTML tags using BeautifulSoup"""
    soup = BeautifulSoup(str(text), 'html.parser')
    return soup.get_text()

def remove_urls(text):
    """Remove URLs from text"""
    text = re.sub(r'http\S+|www\S+|https\S+', '', str(text), flags=re.MULTILINE)
    return text

def remove_special_chars(text):
    """Remove special characters and digits, keep only letters"""
    text = re.sub(r'[^a-zA-Z\s]', ' ', str(text))
    return text

def remove_extra_whitespace(text):
    """Remove extra whitespaces"""
    text = re.sub(r'\s+', ' ', str(text)).strip()
    return text

def preprocess_text(text, remove_stopwords=True, use_lemmatization=True):
    """
    Complete preprocessing pipeline

    Parameters:
    - text: input text
    - remove_stopwords: whether to remove stopwords
    - use_lemmatization: True for lemmatization, False for stemming

    Returns:
    - cleaned text
    """

    text = clean_html(text)

    text = text.lower()

    text = remove_urls(text)

    text = remove_special_chars(text)

    text = remove_extra_whitespace(text)

    tokens = word_tokenize(text)

    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        tokens = [word for word in tokens if word not in stop_words and len(word) > 2]

    if use_lemmatization:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
    else:
        stemmer = PorterStemmer()
        tokens = [stemmer.stem(word) for word in tokens]

    return ' '.join(tokens)

print("✓ Preprocessing functions created")


## Step 3.2: Test Preprocessing on Samples

In [ ]:
print("\n" + "="*80)
print("PREPROCESSING COMPARISON")
print("="*80)

sample_text = train_df[text_column].iloc[0]

print("\nORIGINAL TEXT:")
print(sample_text[:500])

print("\n" + "-"*80)
print("OPTION 1: Stopwords removed + Lemmatization")
print("-"*80)
cleaned_1 = preprocess_text(sample_text, remove_stopwords=True, use_lemmatization=True)
print(cleaned_1[:500])

print("\n" + "-"*80)
print("OPTION 2: Stopwords removed + Stemming")
print("-"*80)
cleaned_2 = preprocess_text(sample_text, remove_stopwords=True, use_lemmatization=False)
print(cleaned_2[:500])

print("\n" + "-"*80)
print("OPTION 3: Stopwords kept + Lemmatization")
print("-"*80)
cleaned_3 = preprocess_text(sample_text, remove_stopwords=False, use_lemmatization=True)
print(cleaned_3[:500])


## Step 3.3: Apply Preprocessing to Entire Dataset

In [ ]:
print("\n" + "="*80)
print("APPLYING PREPROCESSING TO DATASETS")
print("="*80)

print("\n[1/3] Creating Version 1: Stopwords removed + Lemmatization...")
train_df['cleaned_v1'] = train_df[text_column].apply(
    lambda x: preprocess_text(x, remove_stopwords=True, use_lemmatization=True)
)
test_df['cleaned_v1'] = test_df[text_column].apply(
    lambda x: preprocess_text(x, remove_stopwords=True, use_lemmatization=True)
)
print("✓ Version 1 complete")

train_df['cleaned_text'] = train_df['cleaned_v1']
test_df['cleaned_text'] = test_df['cleaned_v1']

print("\n" + "="*80)
print("PREPROCESSING RESULTS")
print("="*80)

print("\nBefore preprocessing:")
print(f"Average text length: {train_df['char_count'].mean():.0f} characters")
print(f"Average word count: {train_df['word_count'].mean():.0f} words")

train_df['cleaned_char_count'] = train_df['cleaned_text'].apply(len)
train_df['cleaned_word_count'] = train_df['cleaned_text'].apply(lambda x: len(x.split()))

print("\nAfter preprocessing:")
print(f"Average text length: {train_df['cleaned_char_count'].mean():.0f} characters")
print(f"Average word count: {train_df['cleaned_word_count'].mean():.0f} words")

print("\n✓ Preprocessing complete!")


# Step-4: Data Splitting

In [ ]:
from sklearn.model_selection import train_test_split

X_full = train_df['cleaned_text']
y_full = train_df['Class']

X_train, X_val, y_train, y_val = train_test_split(
    X_full,
    y_full,
    test_size=0.2,
    random_state=42,
    stratify=y_full
)

X_test = test_df['cleaned_text']
y_test = test_df['Class']

print(f"Training set: {len(X_train)} samples")
print(f"Validation set: {len(X_val)} samples")
print(f"Test set: {len(X_test)} samples")


In [ ]:
print("\nClass distribution in training set:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nClass distribution in validation set:")
print(y_val.value_counts(normalize=True).sort_index())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

y_train.value_counts().plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Training Set Class Distribution', fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

y_val.value_counts().plot(kind='bar', ax=axes[1], color='lightgreen', edgecolor='black')
axes[1].set_title('Validation Set Class Distribution', fontweight='bold')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


# Step-5: Tf-idf Matrix

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True,
    max_features=50000
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

print("\nDone!")
print(f"Training matrix shape: {X_train_tfidf.shape}")
print(f"Validation matrix shape: {X_val_tfidf.shape}")
print(f"Test matrix shape: {X_test_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_)}")


In [ ]:
feature_names = tfidf.get_feature_names_out()
print("\nFirst 20 features:")
print(feature_names[:20])

print("\nLast 20 features:")
print(feature_names[-20:])


In [ ]:
sparsity = 100 * (1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]))
print(f"\nMatrix sparsity: {sparsity:.2f}%")


In [ ]:
import numpy as np

tfidf_scores = np.asarray(X_train_tfidf.mean(axis=0)).flatten()
top_indices = tfidf_scores.argsort()[-20:][::-1]

print("\nTop 20 features by average TF-IDF score:")
for idx in top_indices:
    print(f"{feature_names[idx]}: {tfidf_scores[idx]:.4f}")


In [ ]:
import pickle

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print("\nTF-IDF vectorizer saved!")


# Step-6: Skipgram

In [ ]:
!pip install gensim


In [ ]:
from gensim.models import Word2Vec

def tokenize_for_w2v(text):
    tokens = text.lower().split()
    tokens = [t for t in tokens if len(t) > 1 and not t.isnumeric()]
    return tokens

sentences = [tokenize_for_w2v(text) for text in X_train]

print("Training Skip-gram model (Improved)...")

w2v = Word2Vec(
    sentences=sentences,
    vector_size=200,
    window=7,
    min_count=2,
    workers=4,
    sg=1,
    negative=10,
    sample=1e-3,
    hs=0,
    epochs=50,
    seed=42
)

print("Skip-gram training complete!")
print(f"Vocabulary size: {len(w2v.wv)}")

def show_similar(model, word, topn=5):
    if word in model.wv:
        print(f"\nWords similar to '{word}':")
        print(model.wv.most_similar(word, topn=topn))
    else:
        print(f"\nWord '{word}' not found in vocabulary.")

print("\nTesting Word2Vec embeddings:")
show_similar(w2v, 'love', topn=5)
show_similar(w2v, 'computer', topn=5)

w2v.save('skipgram_model.bin')
w2v.wv.save('skipgram_vectors.kv')

print("\nSkip-gram model saved!")
print("Vectors saved as KeyedVectors!")


# Step-7: ML Models

## Step-7.1: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print("Training Random Forest (Improved)...")

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    max_features=0.3,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    bootstrap=True,
    oob_score=True,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf.fit(X_train_tfidf, y_train)
print("Training done!")

if hasattr(rf, "oob_score_"):
    print(f"\nOOB Score: {rf.oob_score_:.4f}")

train_pred = rf.predict(X_train_tfidf)
train_acc = accuracy_score(y_train, train_pred)
train_f1 = f1_score(y_train, train_pred, average='macro')

print(f"\nTraining Performance:")
print(f"Accuracy: {train_acc:.4f}")
print(f"F1-Score (macro): {train_f1:.4f}")

val_pred = rf.predict(X_val_tfidf)
val_acc = accuracy_score(y_val, val_pred)
val_f1 = f1_score(y_val, val_pred, average='macro')

print(f"\nValidation Performance:")
print(f"Accuracy: {val_acc:.4f}")
print(f"F1-Score (macro): {val_f1:.4f}")

print("\nClassification Report (Validation):")
print(classification_report(y_val, val_pred))

cm = confusion_matrix(y_val, val_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=sorted(y_val.unique()),
    yticklabels=sorted(y_val.unique())
)
plt.title('Random Forest Confusion Matrix (Validation)', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

importances = rf.feature_importances_
indices = importances.argsort()[-20:][::-1]
feature_names = tfidf.get_feature_names_out()

print("\nTop 20 Most Important Features:")
for i, idx in enumerate(indices):
    print(f"{i+1}. {feature_names[idx]}: {importances[idx]:.4f}")

plt.figure(figsize=(10, 6))
plt.barh(range(20), importances[indices])
plt.yticks(range(20), [feature_names[i] for i in indices])
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances', fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

results = {
    'experiment': 'Exp1_TFIDF_RandomForest',
    'model': 'Random Forest',
    'word_rep': 'TF-IDF',
    'hyperparameters': 'n_est=500, max_depth=None, max_feat=0.3, min_split=5, min_leaf=2, class_weight=balanced, oob=True',
    'train_acc': train_acc,
    'train_f1': train_f1,
    'val_acc': val_acc,
    'val_f1': val_f1,
    'test_acc': None,
    'test_f1': None
}

print("\nExperiment 1 Results:")
print(results)

import pickle
with open('rf_model.pkl', 'wb') as f:
    pickle.dump(rf, f)

print("\nModel saved!")


## Step-7.2: Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print("Training Logistic Regression (Improved)...")

lr = LogisticRegression(
    max_iter=3000,
    C=0.8,
    solver='saga',
    penalty='elasticnet',
    l1_ratio=0.2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

lr.fit(X_train_tfidf, y_train)
print("Training done!")

train_pred = lr.predict(X_train_tfidf)
train_acc = accuracy_score(y_train, train_pred)
train_f1 = f1_score(y_train, train_pred, average='macro')

print(f"\nTraining Performance:")
print(f"Accuracy: {train_acc:.4f}")
print(f"F1-Score (macro): {train_f1:.4f}")

val_pred = lr.predict(X_val_tfidf)
val_acc = accuracy_score(y_val, val_pred)
val_f1 = f1_score(y_val, val_pred, average='macro')

print(f"\nValidation Performance:")
print(f"Accuracy: {val_acc:.4f}")
print(f"F1-Score (macro): {val_f1:.4f}")

print("\nClassification Report (Validation):")
print(classification_report(y_val, val_pred))

cm = confusion_matrix(y_val, val_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Greens',
    xticklabels=sorted(y_val.unique()),
    yticklabels=sorted(y_val.unique())
)
plt.title('Logistic Regression Confusion Matrix (Validation)', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

results_lr = {
    'experiment': 'Exp_TFIDF_LogisticRegression',
    'model': 'Logistic Regression',
    'word_rep': 'TF-IDF',
    'hyperparameters': 'C=0.8, max_iter=3000, solver=saga, penalty=elasticnet, l1_ratio=0.2, class_weight=balanced',
    'train_acc': train_acc,
    'train_f1': train_f1,
    'val_acc': val_acc,
    'val_f1': val_f1,
    'test_acc': None,
    'test_f1': None
}

print("\nLogistic Regression Results:")
print(results_lr)

with open('lr_model.pkl', 'wb') as f:
    pickle.dump(lr, f)

print("\nModel saved!")


## Step-7.3: Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print("Training Naive Bayes...")

nb = MultinomialNB(
    alpha=1.0,
    fit_prior=True
)

nb.fit(X_train_tfidf, y_train)
print("Training done!")

train_pred = nb.predict(X_train_tfidf)
train_acc = accuracy_score(y_train, train_pred)
train_f1 = f1_score(y_train, train_pred, average='macro')

print(f"\nTraining Performance:")
print(f"Accuracy: {train_acc:.4f}")
print(f"F1-Score (macro): {train_f1:.4f}")

val_pred = nb.predict(X_val_tfidf)
val_acc = accuracy_score(y_val, val_pred)
val_f1 = f1_score(y_val, val_pred, average='macro')

print(f"\nValidation Performance:")
print(f"Accuracy: {val_acc:.4f}")
print(f"F1-Score (macro): {val_f1:.4f}")

print("\nClassification Report (Validation):")
print(classification_report(y_val, val_pred))

cm = confusion_matrix(y_val, val_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=sorted(y_val.unique()),
            yticklabels=sorted(y_val.unique()))
plt.title('Naive Bayes Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

results_nb = {
    'experiment': 'Exp_TFIDF_NaiveBayes',
    'model': 'Naive Bayes',
    'word_rep': 'TF-IDF',
    'hyperparameters': 'alpha=1.0',
    'train_acc': train_acc,
    'train_f1': train_f1,
    'val_acc': val_acc,
    'val_f1': val_f1,
    'test_acc': None,
    'test_f1': None
}

print("\nNaive Bayes Results:")
print(results_nb)

import pickle

with open('nb_model.pkl', 'wb') as f:
    pickle.dump(nb, f)

print("\nModel saved!")


### Test Set Evaluation - Naive Bayes

In [ ]:
print("\n" + "="*70)
print("NAIVE BAYES - TEST SET EVALUATION")
print("="*70)

test_pred_nb = nb.predict(X_test_tfidf)

test_acc_nb = accuracy_score(y_test, test_pred_nb)
test_f1_nb = f1_score(y_test, test_pred_nb, average='macro')

print(f"\nTest Performance:")
print(f"Accuracy: {test_acc_nb:.4f}")
print(f"F1-Score (macro): {test_f1_nb:.4f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_test, test_pred_nb))

cm_test_nb = confusion_matrix(y_test, test_pred_nb)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_test_nb, annot=True, fmt='d', cmap='Purples',
    xticklabels=sorted(y_test.unique()),
    yticklabels=sorted(y_test.unique())
)
plt.title('Naive Bayes Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

results_nb['test_acc'] = test_acc_nb
results_nb['test_f1'] = test_f1_nb

print("\nUpdated Results:")
print(results_nb)


### Test Set Evaluation - Logistic Regression

In [ ]:
print("\n" + "="*70)
print("LOGISTIC REGRESSION - TEST SET EVALUATION")
print("="*70)

test_pred_lr = lr.predict(X_test_tfidf)

test_acc_lr = accuracy_score(y_test, test_pred_lr)
test_f1_lr = f1_score(y_test, test_pred_lr, average='macro')

print(f"\nTest Performance:")
print(f"Accuracy: {test_acc_lr:.4f}")
print(f"F1-Score (macro): {test_f1_lr:.4f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_test, test_pred_lr))

cm_test_lr = confusion_matrix(y_test, test_pred_lr)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_test_lr, annot=True, fmt='d', cmap='Greens',
    xticklabels=sorted(y_test.unique()),
    yticklabels=sorted(y_test.unique())
)
plt.title('Logistic Regression Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

results_lr['test_acc'] = test_acc_lr
results_lr['test_f1'] = test_f1_lr

print("\nUpdated Results:")
print(results_lr)


### Test Set Evaluation - Random Forest

In [ ]:
print("\n" + "="*70)
print("RANDOM FOREST - TEST SET EVALUATION")
print("="*70)

test_pred_rf = rf.predict(X_test_tfidf)

test_acc_rf = accuracy_score(y_test, test_pred_rf)
test_f1_rf = f1_score(y_test, test_pred_rf, average='macro')

print(f"\nTest Performance:")
print(f"Accuracy: {test_acc_rf:.4f}")
print(f"F1-Score (macro): {test_f1_rf:.4f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_test, test_pred_rf))

cm_test_rf = confusion_matrix(y_test, test_pred_rf)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_test_rf, annot=True, fmt='d', cmap='Blues',
    xticklabels=sorted(y_test.unique()),
    yticklabels=sorted(y_test.unique())
)
plt.title('Random Forest Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

results['test_acc'] = test_acc_rf
results['test_f1'] = test_f1_rf

print("\nUpdated Results:")
print(results)


In [ ]:
models = ['Random Forest', 'Logistic Reg', 'Naive Bayes']
accuracies = [results['val_acc'], results_lr['val_acc'], results_nb['val_acc']]
f1_scores = [results['val_f1'], results_lr['val_f1'], results_nb['val_f1']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(models, accuracies, color=['skyblue', 'lightgreen', 'plum'], edgecolor='black')
axes[0].set_title('ML Models - Validation Accuracy', fontweight='bold')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim([min(accuracies) - 0.05, max(accuracies) + 0.05])
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(accuracies):
    axes[0].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

axes[1].bar(models, f1_scores, color=['skyblue', 'lightgreen', 'plum'], edgecolor='black')
axes[1].set_title('ML Models - Validation F1-Score (Macro)', fontweight='bold')
axes[1].set_ylabel('F1-Score')
axes[1].set_ylim([min(f1_scores) - 0.05, max(f1_scores) + 0.05])
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(f1_scores):
    axes[1].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()


# Step-8: DL Models

## Step-8.1: TF-IDF + Deep Neural Network

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential, layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)
y_test_enc = le.transform(y_test)
n_classes = len(le.classes_)

print(f"Number of classes: {n_classes}")
print(f"Classes: {le.classes_}")

print("="*80)
print("TF-IDF -> TruncatedSVD (for DNN without .toarray())")
print("="*80)

svd = TruncatedSVD(n_components=300, random_state=42)
norm = Normalizer(copy=False)
lsa = make_pipeline(svd, norm)

X_train_tfidf_svd = lsa.fit_transform(X_train_tfidf)
X_val_tfidf_svd   = lsa.transform(X_val_tfidf)
X_test_tfidf_svd  = lsa.transform(X_test_tfidf)

print(f"SVD shapes → Train: {X_train_tfidf_svd.shape} | Val: {X_val_tfidf_svd.shape} | Test: {X_test_tfidf_svd.shape}")
print(f"Explained variance (sum): {svd.explained_variance_ratio_.sum():.4f}")

X_train_dense = X_train_tfidf_svd.astype(np.float32)
X_val_dense   = X_val_tfidf_svd.astype(np.float32)
X_test_dense  = X_test_tfidf_svd.astype(np.float32)

print(f"DNN-ready dense shapes → Train: {X_train_dense.shape} | Val: {X_val_dense.shape} | Test: {X_test_dense.shape}")

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_enc),
    y=y_train_enc
)
class_weights = {i: w for i, w in enumerate(class_weights_array)}
print("\nClass weights:", class_weights)

dnn_model = Sequential([
    layers.Dense(
        256, activation='relu',
        kernel_regularizer=regularizers.l2(0.001),
        input_shape=(X_train_dense.shape[1],)
    ),
    layers.BatchNormalization(),
    layers.Dropout(0.5),

    layers.Dense(
        128, activation='relu',
        kernel_regularizer=regularizers.l2(0.001)
    ),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(n_classes, activation='softmax')
])

optimizer = Adam(learning_rate=3e-4)

dnn_model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(dnn_model.summary())

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

print("Training Deep Neural Network (Improved)...")
history = dnn_model.fit(
    X_train_dense, y_train_enc,
    validation_data=(X_val_dense, y_val_enc),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weights,
    verbose=1
)
print("Training complete!")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

train_pred_dnn = dnn_model.predict(X_train_dense, verbose=0).argmax(axis=1)
val_pred_dnn   = dnn_model.predict(X_val_dense, verbose=0).argmax(axis=1)

train_acc_dnn = accuracy_score(y_train_enc, train_pred_dnn)
train_f1_dnn  = f1_score(y_train_enc, train_pred_dnn, average='macro')
val_acc_dnn   = accuracy_score(y_val_enc, val_pred_dnn)
val_f1_dnn    = f1_score(y_val_enc, val_pred_dnn, average='macro')

print(f"\n{'='*60}")
print(f"DNN RESULTS (IMPROVED)")
print(f"{'='*60}")
print(f"Training   → Accuracy: {train_acc_dnn:.4f} | F1-Macro: {train_f1_dnn:.4f}")
print(f"Validation → Accuracy: {val_acc_dnn:.4f} | F1-Macro: {val_f1_dnn:.4f}")

print("\nClassification Report (Validation):")
print(classification_report(y_val_enc, val_pred_dnn, target_names=le.classes_))

cm_dnn = confusion_matrix(y_val_enc, val_pred_dnn)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_dnn, annot=True, fmt='d', cmap='Oranges',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('Deep NN Confusion Matrix (TF-IDF) - Validation', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

results_dnn = {
    'experiment': 'Exp2_TFIDF_DNN',
    'model': 'Deep Neural Network',
    'word_rep': 'TF-IDF',
    'hyperparameters': '256-128, l2=0.001, dropout=0.5-0.4, lr=3e-4, epochs=100, batch=32, class_weight=balanced',
    'train_acc': train_acc_dnn,
    'train_f1': train_f1_dnn,
    'val_acc': val_acc_dnn,
    'val_f1': val_f1_dnn,
    'test_acc': None,
    'test_f1': None
}

print("\nExperiment 2 Results:")
print(results_dnn)

dnn_model.save('dnn_tfidf_model.keras')
print("\nDNN model saved!")


### Test Set Evaluation - DNN (TF-IDF)

In [ ]:
print("\n" + "="*70)
print("DNN (TF-IDF) - TEST SET EVALUATION")
print("="*70)

test_pred_dnn = dnn_model.predict(X_test_dense, verbose=0).argmax(axis=1)

test_acc_dnn = accuracy_score(y_test_enc, test_pred_dnn)
test_f1_dnn = f1_score(y_test_enc, test_pred_dnn, average='macro')

print(f"\nTest Performance:")
print(f"Accuracy: {test_acc_dnn:.4f}")
print(f"F1-Score (macro): {test_f1_dnn:.4f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_test_enc, test_pred_dnn, target_names=le.classes_))

cm_test_dnn = confusion_matrix(y_test_enc, test_pred_dnn)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_test_dnn, annot=True, fmt='d', cmap='Oranges',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('DNN (TF-IDF) Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

results_dnn['test_acc'] = test_acc_dnn
results_dnn['test_f1'] = test_f1_dnn

print("\nUpdated Results:")
print(results_dnn)


## Step-8.1: Skip-gram + Deep Neural Network

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential, layers
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import regularizers

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

tf.random.set_seed(42)
np.random.seed(42)

def text_to_embedding(text, model):

    vector_size = model.vector_size

    words = text.split()
    vectors = [model.wv[w] for w in words if w in model.wv]

    if len(vectors) == 0:
        return np.zeros(vector_size * 3)

    vectors = np.array(vectors)

    avg = np.mean(vectors, axis=0)
    max_vec = np.max(vectors, axis=0)
    min_vec = np.min(vectors, axis=0)

    return np.concatenate([avg, max_vec, min_vec])

print("Converting texts to Skip-gram embeddings...")

X_train_w2v = np.array([text_to_embedding(text, w2v) for text in X_train])
X_val_w2v   = np.array([text_to_embedding(text, w2v) for text in X_val])
X_test_w2v  = np.array([text_to_embedding(text, w2v) for text in X_test])

print(f"Training embeddings shape: {X_train_w2v.shape}")
print(f"Validation embeddings shape: {X_val_w2v.shape}")
print(f"Test embeddings shape: {X_test_w2v.shape}")

print("\nScaling Skip-gram pooled features (StandardScaler)...")
scaler = StandardScaler()
X_train_w2v = scaler.fit_transform(X_train_w2v)
X_val_w2v   = scaler.transform(X_val_w2v)
X_test_w2v  = scaler.transform(X_test_w2v)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_enc),
    y=y_train_enc
)
class_weights = {i: w for i, w in enumerate(class_weights_array)}
print("Class weights:", class_weights)

feature_size = X_train_w2v.shape[1]

dnn_skipgram = Sequential([
    layers.Dense(
        256, activation='relu',
        kernel_regularizer=regularizers.l2(1e-4),
        input_shape=(feature_size,)
    ),
    BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(
        128, activation='relu',
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(
        64, activation='relu',
        kernel_regularizer=regularizers.l2(1e-4)
    ),
    layers.Dropout(0.3),

    layers.Dense(n_classes, activation='softmax')
])

optimizer = Adam(learning_rate=1e-4)

dnn_skipgram.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(dnn_skipgram.summary())

print("Training Deep Neural Network with Skip-gram (Improved)...")

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

history_skipgram = dnn_skipgram.fit(
    X_train_w2v, y_train_enc,
    validation_data=(X_val_w2v, y_val_enc),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weights,
    verbose=1
)

print("Training complete!")
print(f"Stopped at epoch: {len(history_skipgram.history['loss'])}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_skipgram.history['accuracy'], label='Train Accuracy')
axes[0].plot(history_skipgram.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('DNN Skip-gram - Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_skipgram.history['loss'], label='Train Loss')
axes[1].plot(history_skipgram.history['val_loss'], label='Val Loss')
axes[1].set_title('DNN Skip-gram - Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

train_pred_skip = dnn_skipgram.predict(X_train_w2v, verbose=0).argmax(axis=1)
val_pred_skip   = dnn_skipgram.predict(X_val_w2v, verbose=0).argmax(axis=1)

train_acc_skip = accuracy_score(y_train_enc, train_pred_skip)
train_f1_skip  = f1_score(y_train_enc, train_pred_skip, average='macro')
val_acc_skip   = accuracy_score(y_val_enc, val_pred_skip)
val_f1_skip    = f1_score(y_val_enc, val_pred_skip, average='macro')

print(f"\n{'='*60}")
print(f"DNN + SKIP-GRAM RESULTS (IMPROVED)")
print(f"{'='*60}")
print(f"Training   → Accuracy: {train_acc_skip:.4f} | F1-Macro: {train_f1_skip:.4f}")
print(f"Validation → Accuracy: {val_acc_skip:.4f} | F1-Macro: {val_f1_skip:.4f}")

print("\nClassification Report (Validation):")
print(classification_report(y_val_enc, val_pred_skip, target_names=le.classes_))

cm_skip = confusion_matrix(y_val_enc, val_pred_skip)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_skip, annot=True, fmt='d', cmap='Purples',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('DNN Skip-gram Confusion Matrix (Validation)', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

results_dnn_skipgram = {
    'experiment': 'Exp3_Skipgram_DNN',
    'model': 'Deep Neural Network',
    'word_rep': 'Skip-gram',
    'hyperparameters': '256-128-64, dropout=0.4-0.3-0.3, l2=1e-4, lr=1e-4, epochs=100, batch=32, scaled=StandardScaler, class_weight=balanced',
    'train_acc': train_acc_skip,
    'train_f1': train_f1_skip,
    'val_acc': val_acc_skip,
    'val_f1': val_f1_skip,
    'test_acc': None,
    'test_f1': None
}

print("\nExperiment 3 Results:")
print(results_dnn_skipgram)

dnn_skipgram.save('dnn_skipgram_model.keras')
print("\nModel saved!")


### Test Set Evaluation - DNN (Skip-gram)

In [ ]:
print("\n" + "="*70)
print("DNN (Skip-gram) - TEST SET EVALUATION")
print("="*70)

test_pred_skip = dnn_skipgram.predict(X_test_w2v, verbose=0).argmax(axis=1)

test_acc_skip = accuracy_score(y_test_enc, test_pred_skip)
test_f1_skip = f1_score(y_test_enc, test_pred_skip, average='macro')

print(f"\nTest Performance:")
print(f"Accuracy: {test_acc_skip:.4f}")
print(f"F1-Score (macro): {test_f1_skip:.4f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_test_enc, test_pred_skip, target_names=le.classes_))

cm_test_skip = confusion_matrix(y_test_enc, test_pred_skip)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_test_skip, annot=True, fmt='d', cmap='Purples',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('DNN (Skip-gram) Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

results_dnn_skipgram['test_acc'] = test_acc_skip
results_dnn_skipgram['test_f1'] = test_f1_skip

print("\nUpdated Results:")
print(results_dnn_skipgram)


## Step-8.3: Skip-gram + RNN

In [ ]:
from tensorflow.keras import Sequential, layers
from tensorflow.keras.layers import SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print("="*80)
print("EXPERIMENT: SimpleRNN (FIXED + IMPROVED)")
print("="*80)

tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>", lower=True)
tokenizer.fit_on_texts(X_train)

print("\n[1/6] Converting text to sequences and padding...")

X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_val_sequences   = tokenizer.texts_to_sequences(X_val)
X_test_sequences  = tokenizer.texts_to_sequences(X_test)

calculated_max_length = max(len(seq) for seq in X_train_sequences)
max_length = min(calculated_max_length, 200)
print(f"Calculated max sequence length: {calculated_max_length}")
print(f"Using max length: {max_length}")

X_train_seq = pad_sequences(X_train_sequences, maxlen=max_length, padding='post', truncating='post')
X_val_seq   = pad_sequences(X_val_sequences,   maxlen=max_length, padding='post', truncating='post')
X_test_seq  = pad_sequences(X_test_sequences,  maxlen=max_length, padding='post', truncating='post')

print(f"Padded shapes - Train: {X_train_seq.shape}, Val: {X_val_seq.shape}, Test: {X_test_seq.shape}")

print("\n[2/6] Creating embedding matrix (FIXED)...")

vocab_size = min(5000, len(tokenizer.word_index) + 1)

embedding_dim = w2v.vector_size

embedding_matrix = np.random.normal(scale=0.6, size=(vocab_size, embedding_dim)).astype(np.float32)
embedding_matrix[0] = np.zeros(embedding_dim, dtype=np.float32)

words_found = 0
for word, idx in tokenizer.word_index.items():
    if idx >= vocab_size:
        continue
    if word in w2v.wv:
        embedding_matrix[idx] = w2v.wv[word]
        words_found += 1
    elif word.lower() in w2v.wv:
        embedding_matrix[idx] = w2v.wv[word.lower()]
        words_found += 1

coverage = words_found / vocab_size
print(f"✓ Embedding matrix: {embedding_matrix.shape}")
print(f"✓ Words found in Word2Vec: {words_found}/{vocab_size} (coverage={coverage:.2%})")

print("\n[3/6] Diagnostics (OOV rate)...")
oov_id = tokenizer.word_index.get(tokenizer.oov_token, None)

if oov_id is not None:

    train_tokens_total = sum(len(seq) for seq in X_train_sequences)
    train_oov_total = sum(sum(1 for t in seq if t == oov_id) for seq in X_train_sequences)
    oov_rate = (train_oov_total / train_tokens_total) if train_tokens_total > 0 else 0.0
    print(f"OOV token id: {oov_id}")
    print(f"Training token OOV rate: {oov_rate:.2%}")
else:
    print("OOV token not found in tokenizer.word_index (unexpected).")

print("\n[4/6] Computing class weights...")
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_enc),
    y=y_train_enc
)
class_weights = {i: w for i, w in enumerate(class_weights_array)}
print("Class weights:", class_weights)

print("\n[5/6] Building SimpleRNN model (FIXED)...")

simple_rnn = Sequential([
    layers.Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_length,
        trainable=True,
        mask_zero=True
    ),
    SpatialDropout1D(0.2),

    layers.SimpleRNN(128, dropout=0.3),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(n_classes, activation='softmax')
])

optimizer = Adam(learning_rate=0.001, clipnorm=1.0)

simple_rnn.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(simple_rnn.summary())

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_simplernn.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

print("\n[6/6] Training SimpleRNN...")
history_rnn = simple_rnn.fit(
    X_train_seq, y_train_enc,
    validation_data=(X_val_seq, y_val_enc),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop, reduce_lr, checkpoint],
    class_weight=class_weights,
    verbose=1
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_rnn.history['accuracy'], label='Train Accuracy')
axes[0].plot(history_rnn.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('SimpleRNN - Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_rnn.history['loss'], label='Train Loss')
axes[1].plot(history_rnn.history['val_loss'], label='Val Loss')
axes[1].set_title('SimpleRNN - Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("RNN MODEL + SKIP-GRAM RESULTS (FIXED)")
print("="*60)

train_pred_rnn = simple_rnn.predict(X_train_seq, verbose=0).argmax(axis=1)
val_pred_rnn   = simple_rnn.predict(X_val_seq,   verbose=0).argmax(axis=1)

train_acc_rnn = accuracy_score(y_train_enc, train_pred_rnn)
train_f1_rnn  = f1_score(y_train_enc, train_pred_rnn, average='macro')

val_acc_rnn = accuracy_score(y_val_enc, val_pred_rnn)
val_f1_rnn  = f1_score(y_val_enc, val_pred_rnn, average='macro')

print(f"Training   → Accuracy: {train_acc_rnn:.4f} | F1-Macro: {train_f1_rnn:.4f}")
print(f"Validation → Accuracy: {val_acc_rnn:.4f} | F1-Macro: {val_f1_rnn:.4f}")

print("\nClassification Report (Validation):")
print(classification_report(y_val_enc, val_pred_rnn, target_names=le.classes_))

cm_rnn = confusion_matrix(y_val_enc, val_pred_rnn)
plt.figure(figsize=(12, 10))
sns.heatmap(cm_rnn, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_,
            yticklabels=le.classes_)
plt.title('SimpleRNN Confusion Matrix (Skip-gram) - Validation', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

results_simple_rnn = {
    'experiment': 'Exp3_Skipgram_SimpleRNN',
    'model': 'SimpleRNN',
    'word_rep': 'Skip-gram',
    'hyperparameters': f'vocab=5000, emb_dim={embedding_dim}, rnn_units=128, dropout=0.3, spatial_dropout=0.2, lr=0.001, trainable_emb=True, clipnorm=1.0',
    'train_acc': train_acc_rnn,
    'train_f1': train_f1_rnn,
    'val_acc': val_acc_rnn,
    'val_f1': val_f1_rnn,
    'test_acc': None,
    'test_f1': None
}

print("\nExperiment 3 Results:")
print(results_simple_rnn)

simple_rnn.save('simple_rnn_model.keras')
print("\nSimpleRNN model saved!")


### Test Set Evaluation - SimpleRNN

In [ ]:
print("\n" + "="*70)
print("SimpleRNN - TEST SET EVALUATION")
print("="*70)

test_pred_rnn = simple_rnn.predict(X_test_seq, verbose=0).argmax(axis=1)

test_acc_rnn = accuracy_score(y_test_enc, test_pred_rnn)
test_f1_rnn = f1_score(y_test_enc, test_pred_rnn, average='macro')

print(f"\nTest Performance:")
print(f"Accuracy: {test_acc_rnn:.4f}")
print(f"F1-Score (macro): {test_f1_rnn:.4f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_test_enc, test_pred_rnn, target_names=le.classes_))

cm_test_rnn = confusion_matrix(y_test_enc, test_pred_rnn)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_test_rnn, annot=True, fmt='d', cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('SimpleRNN Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

results_simple_rnn['test_acc'] = test_acc_rnn
results_simple_rnn['test_f1'] = test_f1_rnn

print("\nUpdated Results:")
print(results_simple_rnn)


## Step-8.4: Skip-gram + Bidirectional SimpleRNN


In [ ]:
import numpy as np
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout, Bidirectional, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print("="*80)
print("EXPERIMENT: Bidirectional SimpleRNN (IMPROVED)")
print("="*80)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_enc),
    y=y_train_enc
)
class_weights = {i: w for i, w in enumerate(class_weights_array)}
print("Class weights:", class_weights)

bidirectional_simple_rnn = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_length,
        trainable=True,
        mask_zero=True,
        name='embedding_layer'
    ),
    SpatialDropout1D(0.2),

    Bidirectional(
        SimpleRNN(
            units=64,
            dropout=0.3,
            return_sequences=False
        ),
        name='bidirectional_rnn'
    ),

    Dropout(0.3),
    Dense(n_classes, activation='softmax', name='output_layer')
], name='Bidirectional_SimpleRNN')

optimizer = Adam(learning_rate=0.001, clipnorm=1.0)

bidirectional_simple_rnn.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

bidirectional_simple_rnn.summary()

early_stop_birnn = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_birnn = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

checkpoint_birnn = ModelCheckpoint(
    'best_bidirectional_simplernn.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

print("\nTraining Bidirectional SimpleRNN...")

history_birnn = bidirectional_simple_rnn.fit(
    X_train_seq, y_train_enc,
    validation_data=(X_val_seq, y_val_enc),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop_birnn, reduce_lr_birnn, checkpoint_birnn],
    class_weight=class_weights,
    verbose=1
)

train_pred_birnn = bidirectional_simple_rnn.predict(X_train_seq, verbose=0).argmax(axis=1)
val_pred_birnn   = bidirectional_simple_rnn.predict(X_val_seq, verbose=0).argmax(axis=1)

train_acc_birnn = accuracy_score(y_train_enc, train_pred_birnn)
train_f1_birnn  = f1_score(y_train_enc, train_pred_birnn, average='macro')

val_acc_birnn = accuracy_score(y_val_enc, val_pred_birnn)
val_f1_birnn  = f1_score(y_val_enc, val_pred_birnn, average='macro')

print("\n" + "="*60)
print("BIDIRECTIONAL SIMPLERNN RESULTS")
print("="*60)
print(f"Training   → Accuracy: {train_acc_birnn:.4f} | F1-Macro: {train_f1_birnn:.4f}")
print(f"Validation → Accuracy: {val_acc_birnn:.4f} | F1-Macro: {val_f1_birnn:.4f}")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_birnn.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history_birnn.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].set_title('Bidirectional SimpleRNN - Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_birnn.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history_birnn.history['val_loss'], label='Val Loss', linewidth=2)
axes[1].set_title('Bidirectional SimpleRNN - Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nClassification Report (Validation):")
print(classification_report(y_val_enc, val_pred_birnn, target_names=le.classes_))

cm_birnn = confusion_matrix(y_val_enc, val_pred_birnn)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_birnn, annot=True, fmt='d', cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('Bidirectional SimpleRNN Confusion Matrix (Skip-gram) - Validation', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

results_bidirectional_simple_rnn = {
    'experiment': 'Exp4_Skipgram_BidirectionalSimpleRNN',
    'model': 'Bidirectional SimpleRNN',
    'word_rep': 'Skip-gram',
    'hyperparameters': f'vocab={vocab_size}, emb_dim={embedding_dim}, units=64, spatial_dropout=0.2, dropout=0.3, lr=0.001, clipnorm=1.0, trainable_emb=True, mask_zero=True',
    'train_acc': train_acc_birnn,
    'train_f1': train_f1_birnn,
    'val_acc': val_acc_birnn,
    'val_f1': val_f1_birnn,
    'test_acc': None,
    'test_f1': None
}

print("\nExperiment 4 Results:")
print(results_bidirectional_simple_rnn)

bidirectional_simple_rnn.save('bidirectional_simplernn_model.keras')
print("\nBidirectional SimpleRNN model saved!")


### Test Set Evaluation - Bidirectional SimpleRNN

In [ ]:
print("\n" + "="*70)
print("BIDIRECTIONAL SimpleRNN - TEST SET EVALUATION")
print("="*70)

test_pred_birnn = bidirectional_simple_rnn.predict(X_test_seq, verbose=0).argmax(axis=1)

test_acc_birnn = accuracy_score(y_test_enc, test_pred_birnn)
test_f1_birnn = f1_score(y_test_enc, test_pred_birnn, average='macro')

print(f"\nTest Performance:")
print(f"Accuracy: {test_acc_birnn:.4f}")
print(f"F1-Score (macro): {test_f1_birnn:.4f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_test_enc, test_pred_birnn, target_names=le.classes_))

cm_test_birnn = confusion_matrix(y_test_enc, test_pred_birnn)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_test_birnn, annot=True, fmt='d', cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('Bidirectional SimpleRNN Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

results_bidirectional_simple_rnn['test_acc'] = test_acc_birnn
results_bidirectional_simple_rnn['test_f1'] = test_f1_birnn

print("\nUpdated Results:")
print(results_bidirectional_simple_rnn)


## Step-8.5: Skip-gram + LSTM

In [ ]:
import numpy as np
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print("="*80)
print("EXPERIMENT: LSTM (IMPROVED)")
print("="*80)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_enc),
    y=y_train_enc
)
class_weights = {i: w for i, w in enumerate(class_weights_array)}
print("Class weights:", class_weights)

lstm_model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_length,
        trainable=True,
        mask_zero=True,
        name='embedding_layer'
    ),
    SpatialDropout1D(0.2),

    LSTM(
        units=128,
        dropout=0.3,
        return_sequences=False,
        name='lstm_layer'
    ),

    Dropout(0.3),
    Dense(n_classes, activation='softmax', name='output_layer')
], name='LSTM_Model')

optimizer = Adam(learning_rate=0.001, clipnorm=1.0)

lstm_model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()

early_stop_lstm = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_lstm = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

checkpoint_lstm = ModelCheckpoint(
    'best_lstm.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

print("\nTraining LSTM model...")

history_lstm = lstm_model.fit(
    X_train_seq, y_train_enc,
    validation_data=(X_val_seq, y_val_enc),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop_lstm, reduce_lr_lstm, checkpoint_lstm],
    class_weight=class_weights,
    verbose=1
)

train_pred_lstm = lstm_model.predict(X_train_seq, verbose=0).argmax(axis=1)
val_pred_lstm   = lstm_model.predict(X_val_seq, verbose=0).argmax(axis=1)

train_acc_lstm = accuracy_score(y_train_enc, train_pred_lstm)
train_f1_lstm  = f1_score(y_train_enc, train_pred_lstm, average='macro')

val_acc_lstm = accuracy_score(y_val_enc, val_pred_lstm)
val_f1_lstm  = f1_score(y_val_enc, val_pred_lstm, average='macro')

print("\n" + "="*60)
print("LSTM MODEL RESULTS")
print("="*60)
print(f"Training   → Accuracy: {train_acc_lstm:.4f} | F1-Macro: {train_f1_lstm:.4f}")
print(f"Validation → Accuracy: {val_acc_lstm:.4f} | F1-Macro: {val_f1_lstm:.4f}")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_lstm.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history_lstm.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].set_title('LSTM - Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_lstm.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history_lstm.history['val_loss'], label='Val Loss', linewidth=2)
axes[1].set_title('LSTM - Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nClassification Report (Validation):")
print(classification_report(y_val_enc, val_pred_lstm, target_names=le.classes_))

cm_lstm = confusion_matrix(y_val_enc, val_pred_lstm)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_lstm, annot=True, fmt='d', cmap='Greens',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('LSTM Confusion Matrix (Skip-gram) - Validation', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

results_lstm = {
    'experiment': 'Exp5_Skipgram_LSTM',
    'model': 'LSTM',
    'word_rep': 'Skip-gram',
    'hyperparameters': f'vocab={vocab_size}, emb_dim={embedding_dim}, units=128, spatial_dropout=0.2, dropout=0.3, lr=0.001, clipnorm=1.0, trainable_emb=True, mask_zero=True, epochs<=50',
    'train_acc': train_acc_lstm,
    'train_f1': train_f1_lstm,
    'val_acc': val_acc_lstm,
    'val_f1': val_f1_lstm,
    'test_acc': None,
    'test_f1': None
}

print("\nExperiment 5 Results:")
print(results_lstm)

lstm_model.save('lstm_model.keras')
print("\nLSTM model saved!")


### Test Set Evaluation - LSTM

In [ ]:
print("\n" + "="*70)
print("LSTM - TEST SET EVALUATION")
print("="*70)

test_pred_lstm = lstm_model.predict(X_test_seq, verbose=0).argmax(axis=1)

test_acc_lstm = accuracy_score(y_test_enc, test_pred_lstm)
test_f1_lstm = f1_score(y_test_enc, test_pred_lstm, average='macro')

print(f"\nTest Performance:")
print(f"Accuracy: {test_acc_lstm:.4f}")
print(f"F1-Score (macro): {test_f1_lstm:.4f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_test_enc, test_pred_lstm, target_names=le.classes_))

cm_test_lstm = confusion_matrix(y_test_enc, test_pred_lstm)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_test_lstm, annot=True, fmt='d', cmap='Greens',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('LSTM Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

results_lstm['test_acc'] = test_acc_lstm
results_lstm['test_f1'] = test_f1_lstm

print("\nUpdated Results:")
print(results_lstm)


## Step-8.6: Skip-gram + Bidirectional LSTM


In [ ]:
import numpy as np
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print("="*80)
print("EXPERIMENT: Bidirectional LSTM (IMPROVED)")
print("="*80)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_enc),
    y=y_train_enc
)
class_weights = {i: w for i, w in enumerate(class_weights_array)}
print("Class weights:", class_weights)

bilstm_model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_length,
        trainable=True,
        mask_zero=True,
        name='embedding_layer'
    ),
    SpatialDropout1D(0.2),

    Bidirectional(
        LSTM(
            units=128,
            dropout=0.3,
            return_sequences=False
        ),
        name='bidirectional_lstm'
    ),

    Dropout(0.4),
    Dense(n_classes, activation='softmax', name='output_layer')
], name='Bidirectional_LSTM')

optimizer = Adam(learning_rate=0.001, clipnorm=1.0)

bilstm_model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

bilstm_model.summary()

early_stop_bilstm = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_bilstm = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

checkpoint_bilstm = ModelCheckpoint(
    'best_bilstm.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

print("\nTraining Bidirectional LSTM model...")

history_bilstm = bilstm_model.fit(
    X_train_seq, y_train_enc,
    validation_data=(X_val_seq, y_val_enc),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop_bilstm, reduce_lr_bilstm, checkpoint_bilstm],
    class_weight=class_weights,
    verbose=1
)

train_pred_bilstm = bilstm_model.predict(X_train_seq, verbose=0).argmax(axis=1)
val_pred_bilstm   = bilstm_model.predict(X_val_seq, verbose=0).argmax(axis=1)

train_acc_bilstm = accuracy_score(y_train_enc, train_pred_bilstm)
train_f1_bilstm  = f1_score(y_train_enc, train_pred_bilstm, average='macro')

val_acc_bilstm = accuracy_score(y_val_enc, val_pred_bilstm)
val_f1_bilstm  = f1_score(y_val_enc, val_pred_bilstm, average='macro')

print("\n" + "="*60)
print("BIDIRECTIONAL LSTM RESULTS")
print("="*60)
print(f"Training   → Accuracy: {train_acc_bilstm:.4f} | F1-Macro: {train_f1_bilstm:.4f}")
print(f"Validation → Accuracy: {val_acc_bilstm:.4f} | F1-Macro: {val_f1_bilstm:.4f}")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_bilstm.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history_bilstm.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].set_title('Bidirectional LSTM - Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_bilstm.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history_bilstm.history['val_loss'], label='Val Loss', linewidth=2)
axes[1].set_title('Bidirectional LSTM - Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nClassification Report (Validation):")
print(classification_report(y_val_enc, val_pred_bilstm, target_names=le.classes_))

cm_bilstm = confusion_matrix(y_val_enc, val_pred_bilstm)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_bilstm, annot=True, fmt='d', cmap='Purples',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('Bidirectional LSTM Confusion Matrix (Skip-gram) - Validation', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

results_bilstm = {
    'experiment': 'Exp6_Skipgram_BiLSTM',
    'model': 'Bidirectional LSTM',
    'word_rep': 'Skip-gram',
    'hyperparameters': f'vocab={vocab_size}, emb_dim={embedding_dim}, units=128, spatial_dropout=0.2, dropout=0.3, post_dropout=0.4, lr=0.001, clipnorm=1.0, trainable_emb=True, mask_zero=True, epochs<=50',
    'train_acc': train_acc_bilstm,
    'train_f1': train_f1_bilstm,
    'val_acc': val_acc_bilstm,
    'val_f1': val_f1_bilstm,
    'test_acc': None,
    'test_f1': None
}

print("\nExperiment 6 Results:")
print(results_bilstm)

bilstm_model.save('bilstm_model.keras')
print("\nBidirectional LSTM model saved!")


### Test Set Evaluation - Bidirectional LSTM

In [ ]:
print("\n" + "="*70)
print("BIDIRECTIONAL LSTM - TEST SET EVALUATION")
print("="*70)

test_pred_bilstm = bilstm_model.predict(X_test_seq, verbose=0).argmax(axis=1)

test_acc_bilstm = accuracy_score(y_test_enc, test_pred_bilstm)
test_f1_bilstm = f1_score(y_test_enc, test_pred_bilstm, average='macro')

print(f"\nTest Performance:")
print(f"Accuracy: {test_acc_bilstm:.4f}")
print(f"F1-Score (macro): {test_f1_bilstm:.4f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_test_enc, test_pred_bilstm, target_names=le.classes_))

cm_test_bilstm = confusion_matrix(y_test_enc, test_pred_bilstm)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_test_bilstm, annot=True, fmt='d', cmap='Purples',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('Bidirectional LSTM Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

results_bilstm['test_acc'] = test_acc_bilstm
results_bilstm['test_f1'] = test_f1_bilstm

print("\nUpdated Results:")
print(results_bilstm)


## Step-8.7: Skip-gram + GRU

In [ ]:
import numpy as np
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print("="*80)
print("EXPERIMENT: GRU (IMPROVED)")
print("="*80)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_enc),
    y=y_train_enc
)
class_weights = {i: w for i, w in enumerate(class_weights_array)}
print("Class weights:", class_weights)

gru_model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_length,
        trainable=True,
        mask_zero=True,
        name='embedding_layer'
    ),
    SpatialDropout1D(0.2),

    GRU(
        units=128,
        dropout=0.3,
        return_sequences=False,
        name='gru_layer'
    ),

    Dropout(0.3),
    Dense(n_classes, activation='softmax', name='output_layer')
], name='GRU_Model')

optimizer = Adam(learning_rate=0.001, clipnorm=1.0)

gru_model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

gru_model.summary()

early_stop_gru = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_gru = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

checkpoint_gru = ModelCheckpoint(
    'best_gru.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

print("\nTraining GRU model...")

history_gru = gru_model.fit(
    X_train_seq, y_train_enc,
    validation_data=(X_val_seq, y_val_enc),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop_gru, reduce_lr_gru, checkpoint_gru],
    class_weight=class_weights,
    verbose=1
)

train_pred_gru = gru_model.predict(X_train_seq, verbose=0).argmax(axis=1)
val_pred_gru   = gru_model.predict(X_val_seq, verbose=0).argmax(axis=1)

train_acc_gru = accuracy_score(y_train_enc, train_pred_gru)
train_f1_gru  = f1_score(y_train_enc, train_pred_gru, average='macro')

val_acc_gru = accuracy_score(y_val_enc, val_pred_gru)
val_f1_gru  = f1_score(y_val_enc, val_pred_gru, average='macro')

print("\n" + "="*60)
print("GRU MODEL RESULTS")
print("="*60)
print(f"Training   → Accuracy: {train_acc_gru:.4f} | F1-Macro: {train_f1_gru:.4f}")
print(f"Validation → Accuracy: {val_acc_gru:.4f} | F1-Macro: {val_f1_gru:.4f}")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_gru.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history_gru.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].set_title('GRU - Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_gru.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history_gru.history['val_loss'], label='Val Loss', linewidth=2)
axes[1].set_title('GRU - Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nClassification Report (Validation):")
print(classification_report(y_val_enc, val_pred_gru, target_names=le.classes_))

cm_gru = confusion_matrix(y_val_enc, val_pred_gru)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_gru, annot=True, fmt='d', cmap='Oranges',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('GRU Confusion Matrix (Skip-gram) - Validation', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

results_gru = {
    'experiment': 'Exp7_Skipgram_GRU',
    'model': 'GRU',
    'word_rep': 'Skip-gram',
    'hyperparameters': f'vocab={vocab_size}, emb_dim={embedding_dim}, units=128, spatial_dropout=0.2, dropout=0.3, post_dropout=0.3, lr=0.001, clipnorm=1.0, trainable_emb=True, mask_zero=True, epochs<=50',
    'train_acc': train_acc_gru,
    'train_f1': train_f1_gru,
    'val_acc': val_acc_gru,
    'val_f1': val_f1_gru,
    'test_acc': None,
    'test_f1': None
}

print("\nExperiment 7 Results:")
print(results_gru)

gru_model.save('gru_model.keras')
print("\nGRU model saved!")


### Test Set Evaluation - GRU

In [ ]:
print("\n" + "="*70)
print("GRU - TEST SET EVALUATION")
print("="*70)

test_pred_gru = gru_model.predict(X_test_seq, verbose=0).argmax(axis=1)

test_acc_gru = accuracy_score(y_test_enc, test_pred_gru)
test_f1_gru = f1_score(y_test_enc, test_pred_gru, average='macro')

print(f"\nTest Performance:")
print(f"Accuracy: {test_acc_gru:.4f}")
print(f"F1-Score (macro): {test_f1_gru:.4f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_test_enc, test_pred_gru, target_names=le.classes_))

cm_test_gru = confusion_matrix(y_test_enc, test_pred_gru)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_test_gru, annot=True, fmt='d', cmap='Oranges',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('GRU Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

results_gru['test_acc'] = test_acc_gru
results_gru['test_f1'] = test_f1_gru

print("\nUpdated Results:")
print(results_gru)


## Step-8.8: Skip-gram + Bidirectional GRU

In [ ]:
import numpy as np
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout, Bidirectional, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print("="*80)
print("EXPERIMENT: Bidirectional GRU (IMPROVED)")
print("="*80)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_enc),
    y=y_train_enc
)
class_weights = {i: w for i, w in enumerate(class_weights_array)}
print("Class weights:", class_weights)

bigru_model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_length,
        trainable=True,
        mask_zero=True,
        name='embedding_layer'
    ),
    SpatialDropout1D(0.2),

    Bidirectional(
        GRU(
            units=128,
            dropout=0.3,
            return_sequences=False
        ),
        name='bidirectional_gru'
    ),

    Dropout(0.4),
    Dense(n_classes, activation='softmax', name='output_layer')
], name='Bidirectional_GRU')

optimizer = Adam(learning_rate=0.001, clipnorm=1.0)

bigru_model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

bigru_model.summary()

early_stop_bigru = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_bigru = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

checkpoint_bigru = ModelCheckpoint(
    'best_bigru.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

print("\nTraining Bidirectional GRU model...")

history_bigru = bigru_model.fit(
    X_train_seq, y_train_enc,
    validation_data=(X_val_seq, y_val_enc),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop_bigru, reduce_lr_bigru, checkpoint_bigru],
    class_weight=class_weights,
    verbose=1
)

train_pred_bigru = bigru_model.predict(X_train_seq, verbose=0).argmax(axis=1)
val_pred_bigru   = bigru_model.predict(X_val_seq, verbose=0).argmax(axis=1)

train_acc_bigru = accuracy_score(y_train_enc, train_pred_bigru)
train_f1_bigru  = f1_score(y_train_enc, train_pred_bigru, average='macro')

val_acc_bigru = accuracy_score(y_val_enc, val_pred_bigru)
val_f1_bigru  = f1_score(y_val_enc, val_pred_bigru, average='macro')

print("\n" + "="*60)
print("BIDIRECTIONAL GRU RESULTS")
print("="*60)
print(f"Training   → Accuracy: {train_acc_bigru:.4f} | F1-Macro: {train_f1_bigru:.4f}")
print(f"Validation → Accuracy: {val_acc_bigru:.4f} | F1-Macro: {val_f1_bigru:.4f}")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_bigru.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history_bigru.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].set_title('Bidirectional GRU - Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_bigru.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history_bigru.history['val_loss'], label='Val Loss', linewidth=2)
axes[1].set_title('Bidirectional GRU - Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nClassification Report (Validation):")
print(classification_report(y_val_enc, val_pred_bigru, target_names=le.classes_))

cm_bigru = confusion_matrix(y_val_enc, val_pred_bigru)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_bigru, annot=True, fmt='d', cmap='Reds',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('Bidirectional GRU Confusion Matrix (Skip-gram) - Validation', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

results_bigru = {
    'experiment': 'Exp8_Skipgram_BiGRU',
    'model': 'Bidirectional GRU',
    'word_rep': 'Skip-gram',
    'hyperparameters': f'vocab={vocab_size}, emb_dim={embedding_dim}, units=128, spatial_dropout=0.2, dropout=0.3, post_dropout=0.4, lr=0.001, clipnorm=1.0, trainable_emb=True, mask_zero=True, epochs<=50',
    'train_acc': train_acc_bigru,
    'train_f1': train_f1_bigru,
    'val_acc': val_acc_bigru,
    'val_f1': val_f1_bigru,
    'test_acc': None,
    'test_f1': None
}

print("\nExperiment 8 Results:")
print(results_bigru)

bigru_model.save('bigru_model.keras')
print("\nBidirectional GRU model saved!")


# Step-10: Final Test Set Comparison

In [ ]:
print("\n" + "="*80)
print("FINAL MODEL COMPARISON - TEST SET RESULTS")
print("="*80)

test_results_summary = []

def add_test_result(experiment_name, word_rep, model_type, train_acc, train_f1, val_acc, val_f1, test_acc, test_f1):
    test_results_summary.append({
        'Experiment': experiment_name,
        'Word Representation': word_rep,
        'Model Type': model_type,
        'Train Accuracy': train_acc,
        'Train F1': train_f1,
        'Val Accuracy': val_acc,
        'Val F1': val_f1,
        'Test Accuracy': test_acc,
        'Test F1': test_f1,
        'Val-Test Gap': abs(val_acc - test_acc)
    })

try:
    add_test_result('Exp1_RF', 'TF-IDF', 'Random Forest',
                   results['train_acc'], results['train_f1'], results['val_acc'], results['val_f1'],
                   test_acc_rf, test_f1_rf)
except: pass

try:
    add_test_result('Exp1_LR', 'TF-IDF', 'Logistic Regression',
                   results_lr['train_acc'], results_lr['train_f1'], results_lr['val_acc'], results_lr['val_f1'],
                   test_acc_lr, test_f1_lr)
except: pass

try:
    add_test_result('Exp1_NB', 'TF-IDF', 'Naive Bayes',
                   results_nb['train_acc'], results_nb['train_f1'], results_nb['val_acc'], results_nb['val_f1'],
                   test_acc_nb, test_f1_nb)
except: pass

try:
    add_test_result('Exp2_DNN_TFIDF', 'TF-IDF', 'Deep Neural Network',
                   train_acc_dnn, train_f1_dnn, val_acc_dnn, val_f1_dnn, test_acc_dnn, test_f1_dnn)
except: pass

try:
    add_test_result('Exp3_DNN_Skip', 'Skip-gram', 'Deep Neural Network',
                   train_acc_skip, train_f1_skip, val_acc_skip, val_f1_skip, test_acc_skip, test_f1_skip)
except: pass

try:
    add_test_result('Exp4_SimpleRNN', 'Skip-gram', 'SimpleRNN',
                   train_acc_rnn, train_f1_rnn, val_acc_rnn, val_f1_rnn, test_acc_rnn, test_f1_rnn)
except: pass

try:
    add_test_result('Exp5_BiRNN', 'Skip-gram', 'Bidirectional SimpleRNN',
                   train_acc_birnn, train_f1_birnn, val_acc_birnn, val_f1_birnn, test_acc_birnn, test_f1_birnn)
except: pass

try:
    add_test_result('Exp6_LSTM', 'Skip-gram', 'LSTM',
                   train_acc_lstm, train_f1_lstm, val_acc_lstm, val_f1_lstm, test_acc_lstm, test_f1_lstm)
except: pass

try:
    add_test_result('Exp7_BiLSTM', 'Skip-gram', 'Bidirectional LSTM',
                   train_acc_bilstm, train_f1_bilstm, val_acc_bilstm, val_f1_bilstm, test_acc_bilstm, test_f1_bilstm)
except: pass

try:
    add_test_result('Exp8_GRU', 'Skip-gram', 'GRU',
                   train_acc_gru, train_f1_gru, val_acc_gru, val_f1_gru, test_acc_gru, test_f1_gru)
except: pass

try:
    add_test_result('Exp9_BiGRU', 'Skip-gram', 'Bidirectional GRU',
                   train_acc_bigru, train_f1_bigru, val_acc_bigru, val_f1_bigru, test_acc_bigru, test_f1_bigru)
except: pass

test_results_df = pd.DataFrame(test_results_summary)

test_results_df = test_results_df.sort_values('Test Accuracy', ascending=False)

print("\nCOMPREHENSIVE RESULTS TABLE (sorted by Test Accuracy):")
print(test_results_df.to_string(index=False))
print("="*80)

best_test_model = test_results_df.iloc[0]

print("\n" + "="*80)
print("BEST MODEL ON TEST SET")
print("="*80)
print(f"Model: {best_test_model['Model Type']}")
print(f"Word Representation: {best_test_model['Word Representation']}")
print(f"Test Accuracy: {best_test_model['Test Accuracy']:.4f}")
print(f"Test F1-Score (Macro): {best_test_model['Test F1']:.4f}")
print(f"Validation Accuracy: {best_test_model['Val Accuracy']:.4f}")
print(f"Val-Test Gap: {best_test_model['Val-Test Gap']:.4f}")
print("="*80)

test_results_df.to_csv('final_model_comparison_with_test.csv', index=False)
print("\nResults saved to 'final_model_comparison_with_test.csv'")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

ax = axes[0, 0]
test_sorted = test_results_df.sort_values('Test Accuracy')
colors = ['red' if x < 0.65 else 'yellow' if x < 0.75 else 'green'
          for x in test_sorted['Test Accuracy']]
ax.barh(test_sorted['Model Type'], test_sorted['Test Accuracy'], color=colors, alpha=0.7)
ax.set_xlabel('Test Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Model Comparison: Test Accuracy', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
for i, v in enumerate(test_sorted['Test Accuracy']):
    ax.text(v + 0.01, i, f'{v:.4f}', va='center')

ax = axes[0, 1]
f1_sorted = test_results_df.sort_values('Test F1')
colors_f1 = ['red' if x < 0.65 else 'yellow' if x < 0.75 else 'green'
             for x in f1_sorted['Test F1']]
ax.barh(f1_sorted['Model Type'], f1_sorted['Test F1'], color=colors_f1, alpha=0.7)
ax.set_xlabel('Test F1-Score (Macro)', fontsize=12, fontweight='bold')
ax.set_title('Model Comparison: Test F1-Score', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
for i, v in enumerate(f1_sorted['Test F1']):
    ax.text(v + 0.01, i, f'{v:.4f}', va='center')

ax = axes[1, 0]
x = range(len(test_results_df))
width = 0.25
ax.bar([i - width for i in x], test_results_df['Train Accuracy'], width,
       label='Train', alpha=0.8)
ax.bar([i for i in x], test_results_df['Val Accuracy'], width,
       label='Validation', alpha=0.8)
ax.bar([i + width for i in x], test_results_df['Test Accuracy'], width,
       label='Test', alpha=0.8)
ax.set_xlabel('Models', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Train vs Validation vs Test Accuracy', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(test_results_df['Model Type'], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

ax = axes[1, 1]
gap_sorted = test_results_df.sort_values('Val-Test Gap')
colors_gap = ['green' if x < 0.03 else 'yellow' if x < 0.05 else 'red'
              for x in gap_sorted['Val-Test Gap']]
ax.barh(gap_sorted['Model Type'], gap_sorted['Val-Test Gap'], color=colors_gap, alpha=0.7)
ax.set_xlabel('Val-Test Gap (absolute difference)', fontsize=12, fontweight='bold')
ax.set_title('Generalization Analysis (Val-Test Gap)', fontsize=14, fontweight='bold')
ax.axvline(x=0.03, color='green', linestyle='--', alpha=0.5, label='Good (<0.03)')
ax.axvline(x=0.05, color='orange', linestyle='--', alpha=0.5, label='Acceptable (<0.05)')
ax.legend()
ax.grid(axis='x', alpha=0.3)
for i, v in enumerate(gap_sorted['Val-Test Gap']):
    ax.text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()


### Test Set Evaluation - Bidirectional GRU

In [ ]:
print("\n" + "="*70)
print("BIDIRECTIONAL GRU - TEST SET EVALUATION")
print("="*70)

test_pred_bigru = bigru_model.predict(X_test_seq, verbose=0).argmax(axis=1)

test_acc_bigru = accuracy_score(y_test_enc, test_pred_bigru)
test_f1_bigru = f1_score(y_test_enc, test_pred_bigru, average='macro')

print(f"\nTest Performance:")
print(f"Accuracy: {test_acc_bigru:.4f}")
print(f"F1-Score (macro): {test_f1_bigru:.4f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_test_enc, test_pred_bigru, target_names=le.classes_))

cm_test_bigru = confusion_matrix(y_test_enc, test_pred_bigru)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_test_bigru, annot=True, fmt='d', cmap='Reds',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('Bidirectional GRU Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

results_bigru['test_acc'] = test_acc_bigru
results_bigru['test_f1'] = test_f1_bigru

print("\nUpdated Results:")
print(results_bigru)
